<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/09-user-defined-functions/00-python-udfs.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)

print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))


# 9.0 — Python UDFs: mechanics and cost

The escape hatch, at its slowest. We declare UDFs three ways, watch
`BatchEvalPython` appear in the plan, race one against a native
function, and step on all three silent traps on purpose.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("9.0-python-udfs")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

orders = (
    spark.read.csv(f"{DATA_DIR}/orders.csv", header=True, inferSchema=True)
    .withColumn("revenue", F.round(col("quantity") * col("unit_price"), 2))
)
orders.select("order_id", "customer_id", "product", "quantity", "revenue").show(5)

## Declaring a UDF

Wrapper, decorator, SQL registration. The declared return type is a
promise you make; Spark won't check it (see trap #1 below).

In [ ]:
# 1. wrap a plain function
def clean_code(s):
    return s.strip().upper().replace(" ", "_") if s else None

clean_udf = F.udf(clean_code, "string")

# 2. decorator form
@F.udf("double")
def unit_margin(price, qty):
    if price is None or qty is None:
        return None
    return float(price * qty * 0.27)     # pretend margin rule

(orders
 .withColumn("code", clean_udf("product"))
 .withColumn("margin", unit_margin("unit_price", "quantity"))
 .select("product", "code", "unit_price", "quantity", "margin")
 .show(5))

In [ ]:
# 3. register for the SQL side too
orders.createOrReplaceTempView("orders_v")
spark.udf.register("clean_code", clean_code, "string")
spark.sql("SELECT product, clean_code(product) AS code FROM orders_v").show(5)

## The plan: BatchEvalPython, and a codegen fence

Compare the two plans. The native version lives inside a codegen
stage; the UDF forces a hop out of the JVM that nothing can cross.

In [ ]:
upper_udf = F.udf(lambda s: s.upper() if s else None, "string")

orders.withColumn("u", F.upper("product")).explain()     # one codegen region
orders.withColumn("u", upper_udf("product")).explain()   # BatchEvalPython node

## The race: native vs UDF, same logic

Two lines that look equally innocent. The gap is pickle + one
interpreter call per row. (Run the cell twice; the first run pays
one-time warm-up.)

In [ ]:
import time

def bench(label, df):
    start = time.perf_counter()
    df.write.format("noop").mode("overwrite").save()   # runs everything, writes nothing
    print(f"{label:>16}: {time.perf_counter() - start:5.2f} s")

big = spark.range(2_000_000).withColumn(
    "s", F.concat(F.lit("code-"), (col("id") % 997).cast("string")))

bench("native upper", big.select(F.upper("s")))
bench("python udf",   big.select(upper_udf("s")))

## Trap #1 — wrong declared type = silent nulls

The function below returns a float; the declaration says int.
No error. No warning. Just a dead column.

In [ ]:
bad = F.udf(lambda x: None if x is None else x / 2, "int")     # / returns float!
good = F.udf(lambda x: None if x is None else x // 2, "int")   # int division

(orders
 .withColumn("bad_half", bad("quantity"))
 .withColumn("good_half", good("quantity"))
 .select("quantity", "bad_half", "good_half")
 .show(5))
# bad_half: all null. The declared type is a contract, not a cast.

## Trap #2 — nulls arrive as None

`F.upper(NULL)` is NULL for free. Your UDF gets a real `None` and
dies mid-stage. orders.csv has two null quantities waiting for us.

In [ ]:
double_it = F.udf(lambda q: q * 2, "int")            # no None guard

try:
    orders.withColumn("q2", double_it("quantity")).collect()
except Exception as e:
    print(type(e).__name__)
    hits = [ln for ln in str(e).splitlines() if "TypeError" in ln]
    print(hits[0] if hits else str(e).splitlines()[0])

# fix: handle None FIRST, in every UDF
safe = F.udf(lambda q: None if q is None else q * 2, "int")
orders.withColumn("q2", safe("quantity")).select("quantity", "q2").show(5)

## Trap #3 — no evaluation-order guarantee

Use the UDF column in a later filter and Catalyst may compute the
UDF for the filter *and* the projection — or reorder either past a
"protective" filter. Count how often the lambda appears in your
plan; `.asNondeterministic()` is the opt-out (and blocks those
optimizations).

In [ ]:
u = F.udf(lambda s: s.upper() if s else None, "string")

plan_df = orders.withColumn("u", u("country")).where(col("u") == "DE")
plan_df.explain(True)   # how many times does <lambda>(country) appear?

u_nd = u.asNondeterministic()
orders.withColumn("u", u_nd("country")).where(col("u") == "DE").explain()
# the rule either way: a UDF must tolerate EVERY raw value, not
# post-filter values — nothing guarantees the filter ran first.

## Exercises

1. Write a UDF that parses `sample_logs.txt`-style `key=value`
   tokens into just the value, then race it against the native
   `F.regexp_extract` version on 1M generated lines.
2. Take `unit_margin` and change its declaration to `"string"`.
   Predict the output column before running. Now return
   `str(price)` with declaration `"double"`. Predict again.
3. Prove trap #3 to yourself: build a UDF that raises on negative
   inputs, a column that's negative on 10% of rows, and a filter
   that removes negatives *before* the UDF in your code. Does it
   survive `.collect()`? Does it survive with the filter *after* a
   `.cache()` boundary? Explain what you observe from the plans.
4. Register `unit_margin` for SQL and rewrite the margin query in
   pure SQL. Check both plans have the same BatchEvalPython shape.

In [ ]:
# your exercise code here